# 02 — Relationship Analysis

**Project:** Retail Intelligence — Customer, Sales & Demand Analytics
**Phase:** 1 — Data Understanding (Task 2)
**Input:** the 9 raw Olist CSV files in `data/raw/`
**Builds on:** `01_dataset_inventory.ipynb`

## Objective

Before any cleaning or integration, establish **from the data itself**:

1. which column(s) uniquely identify a row in each table (primary keys),
2. which columns link tables together, and how completely they link (foreign keys / referential integrity),
3. how many rows on each side of a link correspond to each other (cardinality),
4. which irregularities in the keys will affect integration,
5. what actually happens to row counts and totals when the tables are joined.

## Ground rules

| Rule | How it is enforced |
|---|---|
| Raw files are never modified | Read-only access; SHA-256 fingerprints are taken before loading and compared at the end |
| No cleaning, de-duplication, filling or identifier changes | No `drop_duplicates`, `fillna`, `strip` or `astype` is applied to any loaded table. Joins and filters produce temporary in-memory results only |
| Nothing is assumed | Every key and relationship below is **tested**. A column named `*_id` is a candidate, never a given |
| No processed datasets are created | Nothing is written to `data/processed/`. Section 9 checks this |
| Reproducible | Fixed relative paths, deterministic computations (no sampling or randomness), and every number comes from code in this notebook |

## 1. Dataset loading

### 1.1 Environment and paths

Paths are relative to the project root, which is found by walking up from the working directory to the first folder that contains `data/raw/`. The notebook therefore runs the same way from the project root or from `notebooks/`.

In [1]:
import hashlib
import sys
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(f"No project root containing data/raw/ above {start}")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Python      : {sys.version.split()[0]}  ({sys.executable})")
print(f"pandas      : {pd.__version__}")
print(f"In .venv    : {'.venv' in Path(sys.prefix).parts}")
print(f"Project root: {PROJECT_ROOT}")

Python      : 3.12.10  (C:\Users\Admin\Desktop\retail-intelligence\.venv\Scripts\python.exe)
pandas      : 3.1.0
In .venv    : True
Project root: C:\Users\Admin\Desktop\retail-intelligence


### 1.2 Fingerprint the raw files before reading them

Each file's size, modification time and SHA-256 hash are recorded so that Section 9 can show the files were not changed. A snapshot of `data/processed/` is also taken to show this notebook writes nothing there.

In [2]:
FILES = {  # short alias -> raw file name
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

def fingerprint(path: Path) -> dict:
    sha = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            sha.update(chunk)
    st = path.stat()
    return {"file": path.name, "size_bytes": st.st_size, "modified_ns": st.st_mtime_ns, "sha256": sha.hexdigest()}

fingerprints_before = pd.DataFrame([fingerprint(RAW_DIR / f) for f in FILES.values()]).set_index("file")
processed_before = sorted(p.name for p in PROCESSED_DIR.iterdir()) if PROCESSED_DIR.exists() else []
display(fingerprints_before[["size_bytes", "sha256"]])

,size_bytes,sha256
file,,
olist_customers_dataset.csv,9033957,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2
olist_geolocation_dataset.csv,61273883,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6
olist_orders_dataset.csv,17654914,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf
olist_order_items_dataset.csv,15438671,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279
olist_order_payments_dataset.csv,5777138,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0
olist_order_reviews_dataset.csv,14451670,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7
olist_products_dataset.csv,2379446,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc
olist_sellers_dataset.csv,174703,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e
product_category_name_translation.csv,2613,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278


### 1.3 Load each file once

**Technical decision: use pandas' default type inference (no overrides).** This matches notebook 01, so both notebooks look at the same in-memory data. For relationship testing what matters is that:

- identifier columns are loaded **exactly as stored** (text, not converted to numbers), and
- the columns on both sides of every link have the **same data type**, so a value comparison means the same thing on both sides.

Both conditions are checked in 1.4 and 3.1 instead of being taken for granted. Loading errors are caught per file and reported. The DataFrames are kept in `datasets` and reused everywhere, so each file is read only once.

In [3]:
datasets: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}

for alias, fname in FILES.items():
    try:
        datasets[alias] = pd.read_csv(RAW_DIR / fname)
    except FileNotFoundError:
        load_errors[alias] = "file not found"
    except pd.errors.EmptyDataError:
        load_errors[alias] = "file is empty"
    except pd.errors.ParserError as exc:
        load_errors[alias] = f"parse error: {exc}"
    except UnicodeDecodeError as exc:
        load_errors[alias] = f"encoding error: {exc}"
    except Exception as exc:
        load_errors[alias] = f"{type(exc).__name__}: {exc}"

assert not load_errors, f"Load errors: {load_errors}"
display(pd.DataFrame({a: {"file": FILES[a], "rows": len(d), "columns": d.shape[1]} for a, d in datasets.items()}).T)

,file,rows,columns
customers,olist_customers_dataset.csv,99441,5
geolocation,olist_geolocation_dataset.csv,1000163,5
orders,olist_orders_dataset.csv,99441,8
order_items,olist_order_items_dataset.csv,112650,7
payments,olist_order_payments_dataset.csv,103886,5
reviews,olist_order_reviews_dataset.csv,99224,7
products,olist_products_dataset.csv,32951,9
sellers,olist_sellers_dataset.csv,3095,4
translation,product_category_name_translation.csv,71,2


### 1.4 Check that identifier columns were loaded as stored

Every `*_id` column should be text. These are 32-character hexadecimal hashes in this dataset, and that format is checked here rather than assumed. If any identifier had been read as a number (for example an all-digit hash), leading zeros would be lost and joins would silently fail.

In [4]:
id_format_rows = []
for alias, df in datasets.items():
    for col in [c for c in df.columns if c.endswith("_id") and c != "order_item_id"]:
        s = df[col].dropna()
        id_format_rows.append({
            "table": alias, "column": col, "dtype": str(df[col].dtype),
            "is_text": pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]),
            "all_32_char_hex": bool(s.astype(str).str.fullmatch(r"[0-9a-f]{32}").all()),
        })
id_format_df = pd.DataFrame(id_format_rows)
display(id_format_df)
assert id_format_df["is_text"].all() and id_format_df["all_32_char_hex"].all()
print("All identifier columns are text and 32-char lowercase hex — loaded unaltered.")

,table,column,dtype,is_text,all_32_char_hex
0,customers,customer_id,str,True,True
1,customers,customer_unique_id,str,True,True
2,orders,order_id,str,True,True
3,orders,customer_id,str,True,True
4,order_items,order_id,str,True,True
5,order_items,product_id,str,True,True
6,order_items,seller_id,str,True,True
7,payments,order_id,str,True,True
8,reviews,review_id,str,True,True
9,reviews,order_id,str,True,True


All identifier columns are text and 32-char lowercase hex — loaded unaltered.


## 2. Primary key analysis

A column, or a combination of columns, can be a **primary key** only if it is (a) **unique** across all rows and (b) **never missing**. Two complementary approaches are used:

1. **Unbiased scan (2.1):** every column of every table is tested for uniqueness and completeness, whatever its name. For tables with no single-column key, every **pair** of columns is also tested. This finds key candidates from the data rather than from naming conventions.
2. **Focused assessment (2.2):** the plausible candidates (including composites) are profiled with the five required metrics.

**Metric definitions**

| Metric | Definition |
|---|---|
| `total_records` | rows in the table |
| `unique_values` | distinct key values (or distinct combinations for composites) |
| `duplicated_values` | rows whose key value repeats an earlier row's value (`df.duplicated(subset)`). Equals `total_records − unique_values` |
| `rows_in_duplicate_groups` | all rows that share their key value with at least one other row |
| `missing_values` | rows where any key column is missing |
| `uniqueness_pct` | `unique_values / total_records × 100` |
| `valid_pk` | `duplicated_values == 0` **and** `missing_values == 0` |

In [5]:
def key_profile(table: str, cols) -> dict:
    cols = [cols] if isinstance(cols, str) else list(cols)
    df = datasets[table]
    total = len(df)
    dup = int(df.duplicated(subset=cols).sum())
    missing = int(df[cols].isna().any(axis=1).sum())
    return {
        "table": table,
        "candidate_key": " + ".join(cols),
        "n_cols": len(cols),
        "total_records": total,
        "unique_values": total - dup,
        "duplicated_values": dup,
        "rows_in_duplicate_groups": int(df.duplicated(subset=cols, keep=False).sum()),
        "missing_values": missing,
        "uniqueness_pct": round((total - dup) / total * 100, 4),
        "valid_pk": dup == 0 and missing == 0,
    }

### 2.1 Unbiased key discovery

All single columns are tested. For tables where no single column qualifies, all 2-column combinations are tested as well. The table shows only the combinations that **qualify** as keys.

In [6]:
discovery = []
for table, df in datasets.items():
    singles = [key_profile(table, c) for c in df.columns]
    valid_singles = [r for r in singles if r["valid_pk"]]
    discovery += valid_singles
    if not valid_singles:  # escalate to pairs only where needed
        discovery += [r for r in (key_profile(table, pair) for pair in combinations(df.columns, 2)) if r["valid_pk"]]

discovery_df = pd.DataFrame(discovery)
display(discovery_df[["table", "candidate_key", "n_cols", "total_records", "unique_values", "valid_pk"]])

no_key = sorted(set(datasets) - set(discovery_df["table"]))
print("Tables with no single- or two-column key:", no_key or "none")

,table,candidate_key,n_cols,total_records,unique_values,valid_pk
0,customers,customer_id,1,99441,99441,True
1,orders,order_id,1,99441,99441,True
2,orders,customer_id,1,99441,99441,True
3,order_items,order_id + order_item_id,2,112650,112650,True
4,payments,order_id + payment_sequential,2,103886,103886,True
5,reviews,review_id + order_id,2,99224,99224,True
6,reviews,order_id + review_answer_timestamp,2,99224,99224,True
7,products,product_id,1,32951,32951,True
8,sellers,seller_id,1,3095,3095,True
9,translation,product_category_name,1,71,71,True


Tables with no single- or two-column key: ['geolocation']


The scan finds some **coincidental** keys. In `orders`, `customer_id` is unique just as `order_id` is. In `translation`, the English category name is also unique. In `reviews`, `order_id + review_answer_timestamp` happens to be unique too. A unique column is not automatically the table's *identifier*. 2.2 therefore compares candidates side by side, and 2.3 explains each choice using what the columns hold.

`reviews`, `order_items` and `payments` need a composite key. `geolocation` has no key at all, not even all columns together, which 2.2 confirms.

### 2.2 Focused candidate-key assessment

In [7]:
PK_CANDIDATES = {
    "customers":   ["customer_id", "customer_unique_id"],
    "geolocation": ["geolocation_zip_code_prefix",
                    ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
                    list(datasets["geolocation"].columns)],
    "orders":      ["order_id", "customer_id"],
    "order_items": ["order_id", "order_item_id", ["order_id", "order_item_id"],
                    ["order_id", "product_id", "seller_id"]],
    "payments":    ["order_id", ["order_id", "payment_sequential"]],
    "reviews":     ["review_id", "order_id", ["review_id", "order_id"]],
    "products":    ["product_id"],
    "sellers":     ["seller_id"],
    "translation": ["product_category_name", "product_category_name_english"],
}

pk_df = pd.DataFrame([key_profile(t, c) for t, cands in PK_CANDIDATES.items() for c in cands])
display(pk_df.style.apply(lambda r: ["background-color: #d9f2d9" if r.valid_pk else "" for _ in r], axis=1)
        .format({"uniqueness_pct": "{:.4f}"}))

,table,candidate_key,n_cols,total_records,unique_values,duplicated_values,rows_in_duplicate_groups,missing_values,uniqueness_pct,valid_pk
0,customers,customer_id,1,99441,99441,0,0,0,100.0000,True
1,customers,customer_unique_id,1,99441,96096,3345,6342,0,96.6362,False
2,geolocation,geolocation_zip_code_prefix,1,1000163,19015,981148,999120,0,1.9012,False
3,geolocation,geolocation_zip_code_prefix + geolocation_lat + geolocation_lng,3,1000163,720154,280009,411716,0,72.0037,False
4,geolocation,geolocation_zip_code_prefix + geolocation_lat + geolocation_lng + geolocation_city + geolocation_state,5,1000163,738332,261831,390005,0,73.8212,False
5,orders,order_id,1,99441,99441,0,0,0,100.0000,True
6,orders,customer_id,1,99441,99441,0,0,0,100.0000,True
7,order_items,order_id,1,112650,98666,13984,23787,0,87.5863,False
8,order_items,order_item_id,1,112650,21,112629,112649,0,0.0186,False
9,order_items,order_id + order_item_id,2,112650,112650,0,0,0,100.0000,True


### 2.3 Primary key decisions

Each choice uses both the test results above **and** what the columns hold:

- `orders.customer_id` is unique, but it is a *reference* to `customers`. `order_id` is the order's own identifier, and the other order-level tables reference it.
- `translation.product_category_name_english` is unique, but the Portuguese name is what `products` references, so the Portuguese name is the key.
- `geolocation` has **no** valid key. Even the full row repeats, so no combination of columns can identify a row.

In [8]:
PRIMARY_KEYS = {
    "customers":   ["customer_id"],
    "geolocation": None,
    "orders":      ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments":    ["order_id", "payment_sequential"],
    "reviews":     ["review_id", "order_id"],
    "products":    ["product_id"],
    "sellers":     ["seller_id"],
    "translation": ["product_category_name"],
}

pk_assessment = []
for table, key in PRIMARY_KEYS.items():
    if key is None:
        full = key_profile(table, list(datasets[table].columns))
        pk_assessment.append({**full, "candidate_key": "(none — all columns tested)", "type": "none"})
    else:
        pk_assessment.append({**key_profile(table, key), "type": "composite" if len(key) > 1 else "single"})
pk_assessment_df = pd.DataFrame(pk_assessment).set_index("table")
display(pk_assessment_df[["candidate_key", "type", "total_records", "unique_values",
                          "duplicated_values", "missing_values", "uniqueness_pct", "valid_pk"]])

# Every table except geolocation must have a validated key
assert pk_assessment_df.drop(index="geolocation")["valid_pk"].all()
assert not pk_assessment_df.loc["geolocation", "valid_pk"]

,candidate_key,type,total_records,unique_values,duplicated_values,missing_values,uniqueness_pct,valid_pk
table,,,,,,,,
customers,customer_id,single,99441,99441,0,0,100.0000,True
geolocation,(none — all columns tested),none,1000163,738332,261831,0,73.8212,False
orders,order_id,single,99441,99441,0,0,100.0000,True
order_items,order_id + order_item_id,composite,112650,112650,0,0,100.0000,True
payments,order_id + payment_sequential,composite,103886,103886,0,0,100.0000,True
reviews,review_id + order_id,composite,99224,99224,0,0,100.0000,True
products,product_id,single,32951,32951,0,0,100.0000,True
sellers,seller_id,single,3095,3095,0,0,100.0000,True
translation,product_category_name,single,71,71,0,0,100.0000,True


## 3. Foreign key analysis

For each relationship, the **child** column (the referencing side) is tested against the **parent** key (the referenced side).

| Metric | Definition |
|---|---|
| `matched_records` | child rows whose non-missing key exists in the parent |
| `unmatched_records` | child rows whose non-missing key does **not** exist in the parent (orphans) |
| `child_missing` | child rows where the reference is empty (counted separately; neither matched nor orphan) |
| `integrity_pct` | `matched / (matched + unmatched) × 100`, i.e. referential integrity over the non-missing references |
| `distinct_child_ids` | distinct identifiers used in the child column |
| `distinct_ids_referenced` | distinct parent keys that at least one child row references |
| `parent_coverage_pct` | share of parent keys that are referenced at all (shows parents with no children) |

**Technical decision: test the values directly with `Series.isin`, not with a merge.** A merge would multiply rows wherever a key repeats, which is exactly what Section 6 investigates. `isin` measures matches with no fan-out. The parent column must be unique for the link to be a true foreign key, so that is checked too (`parent_key_unique`).

Besides the seven required relationships, two **supplementary** links through zip-code prefix to `geolocation` are tested, because that table has no primary key (Section 2).

In [9]:
RELATIONSHIPS = [  # (id, label, parent, parent_key, child, child_key)
    ("R1", "customers ↔ orders",            "customers",   "customer_id",                 "orders",      "customer_id"),
    ("R2", "orders ↔ order_items",          "orders",      "order_id",                    "order_items", "order_id"),
    ("R3", "orders ↔ payments",             "orders",      "order_id",                    "payments",    "order_id"),
    ("R4", "orders ↔ reviews",              "orders",      "order_id",                    "reviews",     "order_id"),
    ("R5", "order_items ↔ products",        "products",    "product_id",                  "order_items", "product_id"),
    ("R6", "order_items ↔ sellers",         "sellers",     "seller_id",                   "order_items", "seller_id"),
    ("R7", "products ↔ translation",        "translation", "product_category_name",       "products",    "product_category_name"),
    ("S1", "customers ↔ geolocation (zip)", "geolocation", "geolocation_zip_code_prefix", "customers",   "customer_zip_code_prefix"),
    ("S2", "sellers ↔ geolocation (zip)",   "geolocation", "geolocation_zip_code_prefix", "sellers",     "seller_zip_code_prefix"),
]

### 3.1 Type compatibility of every join column pair

A link can only be tested meaningfully if both sides have the same type. Comparing text with numbers would report zero matches without raising an error.

In [10]:
type_check = pd.DataFrame([{
    "id": rid, "child": f"{c}.{ck}", "child_dtype": str(datasets[c][ck].dtype),
    "parent": f"{p}.{pk}", "parent_dtype": str(datasets[p][pk].dtype),
    "compatible": datasets[c][ck].dtype == datasets[p][pk].dtype,
} for rid, _, p, pk, c, ck in RELATIONSHIPS]).set_index("id")
display(type_check)
assert type_check["compatible"].all()

,child,child_dtype,parent,parent_dtype,compatible
id,,,,,
R1,orders.customer_id,str,customers.customer_id,str,True
R2,order_items.order_id,str,orders.order_id,str,True
R3,payments.order_id,str,orders.order_id,str,True
R4,reviews.order_id,str,orders.order_id,str,True
R5,order_items.product_id,str,products.product_id,str,True
R6,order_items.seller_id,str,sellers.seller_id,str,True
R7,products.product_category_name,str,translation.product_category_name,str,True
S1,customers.customer_zip_code_prefix,int64,geolocation.geolocation_zip_code_prefix,int64,True
S2,sellers.seller_zip_code_prefix,int64,geolocation.geolocation_zip_code_prefix,int64,True


### 3.2 Referential integrity tests

In [11]:
def fk_test(rid, label, parent, pk, child, ck) -> dict:
    c = datasets[child][ck]
    p = datasets[parent][pk]
    parent_keys = pd.Index(p.dropna().unique())
    present = c.notna()
    matched = present & c.isin(parent_keys)
    unmatched = present & ~matched
    n_present = int(present.sum())
    return {
        "id": rid, "relationship": label,
        "child → parent": f"{child}.{ck} → {parent}.{pk}",
        "child_rows": len(c),
        "matched_records": int(matched.sum()),
        "unmatched_records": int(unmatched.sum()),
        "child_missing": int((~present).sum()),
        "integrity_pct": round(matched.sum() / n_present * 100, 4) if n_present else float("nan"),
        "distinct_child_ids": int(c.nunique()),
        "distinct_unmatched_ids": int(c[unmatched].nunique()),
        "parent_key_unique": bool(p.is_unique),
        "parent_distinct_ids": len(parent_keys),
        "distinct_ids_referenced": int(parent_keys.isin(c).sum()),
        "parent_coverage_pct": round(parent_keys.isin(c).mean() * 100, 4),
    }

fk_results, fk_errors = [], {}
for rel in RELATIONSHIPS:
    try:
        fk_results.append(fk_test(*rel))
    except Exception as exc:
        fk_errors[rel[0]] = f"{type(exc).__name__}: {exc}"

fk_df = pd.DataFrame(fk_results).set_index("id")
display(fk_df)
print("FK test errors:", fk_errors or "none")

,relationship,child → parent,child_rows,matched_records,unmatched_records,child_missing,integrity_pct,distinct_child_ids,distinct_unmatched_ids,parent_key_unique,parent_distinct_ids,distinct_ids_referenced,parent_coverage_pct
id,,,,,,,,,,,,,
R1,customers ↔ orders,orders.customer_id → customers.customer_id,99441,99441,0,0,100.0000,99441,0,True,99441,99441,100.0000
R2,orders ↔ order_items,order_items.order_id → orders.order_id,112650,112650,0,0,100.0000,98666,0,True,99441,98666,99.2206
R3,orders ↔ payments,payments.order_id → orders.order_id,103886,103886,0,0,100.0000,99440,0,True,99441,99440,99.9990
R4,orders ↔ reviews,reviews.order_id → orders.order_id,99224,99224,0,0,100.0000,98673,0,True,99441,98673,99.2277
R5,order_items ↔ products,order_items.product_id → products.product_id,112650,112650,0,0,100.0000,32951,0,True,32951,32951,100.0000
R6,order_items ↔ sellers,order_items.seller_id → sellers.seller_id,112650,112650,0,0,100.0000,3095,0,True,3095,3095,100.0000
R7,products ↔ translation,products.product_category_name → translation.product_category_name,32951,32328,13,610,99.9598,73,2,True,71,71,100.0000
S1,customers ↔ geolocation (zip),customers.customer_zip_code_prefix → geolocation.geolocation_zip_code_prefix,99441,99163,278,0,99.7204,14994,157,False,19015,14837,78.0279
S2,sellers ↔ geolocation (zip),sellers.seller_zip_code_prefix → geolocation.geolocation_zip_code_prefix,3095,3088,7,0,99.7738,2246,7,False,19015,2239,11.7749


FK test errors: none


### 3.3 What the unmatched and unreferenced values are

Two kinds of gap are listed separately:

- **Orphans:** child values with no parent.
- **Unreferenced parents:** parents with no children. These are not integrity violations, but they matter for inner joins.

In [12]:
gap_rows = []
for rid, label, parent, pk, child, ck in RELATIONSHIPS:
    c, p = datasets[child][ck], datasets[parent][pk]
    orphan_vals = c[c.notna() & ~c.isin(p.dropna())].unique()
    unref_parents = p[~p.isin(c.dropna())].unique()
    gap_rows.append({"id": rid, "relationship": label,
                     "orphan_values": len(orphan_vals),
                     "orphan_examples": list(orphan_vals[:3]),
                     "unreferenced_parent_keys": len(unref_parents)})
display(pd.DataFrame(gap_rows).set_index("id"))

,relationship,orphan_values,orphan_examples,unreferenced_parent_keys
id,,,,
R1,customers ↔ orders,0,[],0
R2,orders ↔ order_items,0,[],775
R3,orders ↔ payments,0,[],1
R4,orders ↔ reviews,0,[],768
R5,order_items ↔ products,0,[],0
R6,order_items ↔ sellers,0,[],0
R7,products ↔ translation,2,"[pc_gamer, portateis_cozinha_e_preparadores_de_alimentos]",0
S1,customers ↔ geolocation (zip),157,"[72300, 11547, 64605]",4178
S2,sellers ↔ geolocation (zip),7,"[82040, 91901, 72580]",16776


In [13]:
# Orders that no order_items / payments / reviews row references — profile by order_status (observation only)
orders = datasets["orders"]
no_child = pd.DataFrame({
    "no_items":    ~orders["order_id"].isin(datasets["order_items"]["order_id"]),
    "no_payments": ~orders["order_id"].isin(datasets["payments"]["order_id"]),
    "no_reviews":  ~orders["order_id"].isin(datasets["reviews"]["order_id"]),
})
display(no_child.sum().rename("orders").to_frame())
display(orders.loc[no_child["no_items"], "order_status"].value_counts().rename("orders_without_items").to_frame())
display(orders.loc[no_child["no_payments"], ["order_id", "order_status", "order_purchase_timestamp"]])

,orders
no_items,775
no_payments,1
no_reviews,768


,orders_without_items
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


,order_id,order_status,order_purchase_timestamp
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38


In [14]:
# Product categories with no row in the translation table
products, translation = datasets["products"], datasets["translation"]
untranslated_mask = products["product_category_name"].notna() & ~products["product_category_name"].isin(translation["product_category_name"])
display(products.loc[untranslated_mask, "product_category_name"].value_counts().rename("products").to_frame())

,products
product_category_name,
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


## 4. Relationship cardinality

Cardinality is **measured**, not assumed. For each relationship between a left table L and a right table R joined on a key:

- `max_L_per_key`: the largest number of L rows that share one key value. A value of 1 means each key appears once on the L side.
- `max_R_per_key`: the same for R.
- **Classification:** each side is "one" if its maximum is 1 and "many" if it is greater than 1. *One-to-many* means one L row relates to many R rows.
- **Optionality** (participation) is reported separately. `L_keys_without_R` counts L keys with no R partner (for example, orders without reviews). `R_rows_without_L` counts R rows with no L partner, including missing references.

The relationships are oriented in the order of the task (e.g. `order_items ↔ products` is read from the order-item side). Three **supplementary** relationships are added because Sections 5 and 6 need them:

- `order_items ↔ payments`, which are linked only indirectly through `order_id`
- the two zip-code links to `geolocation`

In [15]:
CARDINALITY_TESTS = [  # (id, L, L_key, R, R_key)
    ("R1", "customers",   "customer_id",              "orders",      "customer_id"),
    ("R2", "orders",      "order_id",                 "order_items", "order_id"),
    ("R3", "orders",      "order_id",                 "payments",    "order_id"),
    ("R4", "orders",      "order_id",                 "reviews",     "order_id"),
    ("R5", "order_items", "product_id",               "products",    "product_id"),
    ("R6", "order_items", "seller_id",                "sellers",     "seller_id"),
    ("R7", "products",    "product_category_name",    "translation", "product_category_name"),
    ("S1", "customers",   "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("S2", "sellers",     "seller_zip_code_prefix",   "geolocation", "geolocation_zip_code_prefix"),
    ("S3", "order_items", "order_id",                 "payments",    "order_id"),
]

def side(n_max: int) -> str:
    return "one" if n_max <= 1 else "many"

def cardinality(rid, L, lk, R, rk) -> dict:
    l, r = datasets[L][lk], datasets[R][rk]
    common = pd.Index(l.dropna().unique()).intersection(pd.Index(r.dropna().unique()))
    l_counts = l[l.isin(common)].value_counts()
    r_counts = r[r.isin(common)].value_counts()
    l_max, r_max = int(l_counts.max()), int(r_counts.max())
    return {
        "id": rid, "relationship": f"{L} ↔ {R}", "join_key": lk if lk == rk else f"{lk} = {rk}",
        "shared_key_values": len(common),
        "max_L_per_key": l_max, "max_R_per_key": r_max,
        "mean_R_per_key": round(r_counts.mean(), 3),
        "L_keys_without_R": int((~pd.Index(l.dropna().unique()).isin(common)).sum()),
        "R_rows_without_L": int((~r.isin(common)).sum()),
        "cardinality": f"{side(l_max)}-to-{side(r_max)}",
    }

card_df = pd.DataFrame([cardinality(*t) for t in CARDINALITY_TESTS]).set_index("id")
display(card_df)

,relationship,join_key,shared_key_values,max_L_per_key,max_R_per_key,mean_R_per_key,L_keys_without_R,R_rows_without_L,cardinality
id,,,,,,,,,
R1,customers ↔ orders,customer_id,99441,1,1,1.000,0,0,one-to-one
R2,orders ↔ order_items,order_id,98666,1,21,1.142,775,0,one-to-many
R3,orders ↔ payments,order_id,99440,1,29,1.045,1,0,one-to-many
R4,orders ↔ reviews,order_id,98673,1,3,1.006,768,0,one-to-many
R5,order_items ↔ products,product_id,32951,527,1,1.000,0,0,many-to-one
R6,order_items ↔ sellers,seller_id,3095,2033,1,1.000,0,0,many-to-one
R7,products ↔ translation,product_category_name,71,3029,1,1.000,2,0,many-to-one
S1,customers ↔ geolocation,customer_zip_code_prefix = geolocation_zip_code_prefix,14837,142,1146,65.262,157,31864,many-to-many
S2,sellers ↔ geolocation,seller_zip_code_prefix = geolocation_zip_code_prefix,2239,49,965,120.061,7,731347,many-to-many


In [16]:
# How many children per parent — the distributions behind the 'many' sides
dist = {
    "order_items per order": datasets["order_items"]["order_id"].value_counts(),
    "payments per order":    datasets["payments"]["order_id"].value_counts(),
    "reviews per order":     datasets["reviews"]["order_id"].value_counts(),
    "orders per review_id":  datasets["reviews"]["review_id"].value_counts(),
    "items per product":     datasets["order_items"]["product_id"].value_counts(),
    "items per seller":      datasets["order_items"]["seller_id"].value_counts(),
}
display(pd.DataFrame({k: v.describe(percentiles=[.5, .9, .99]) for k, v in dist.items()}).T.round(2))

,count,mean,std,min,50%,90%,99%,max
order_items per order,98666.0,1.14,0.54,1.0,1.0,1.0,3.00,21.0
payments per order,99440.0,1.04,0.38,1.0,1.0,1.0,2.00,29.0
reviews per order,98673.0,1.01,0.08,1.0,1.0,1.0,1.00,3.0
orders per review_id,98410.0,1.01,0.09,1.0,1.0,1.0,1.00,3.0
items per product,32951.0,3.42,10.62,1.0,1.0,6.0,33.00,527.0
items per seller,3095.0,36.40,119.19,1.0,8.0,80.0,431.24,2033.0


### 4.1 Expected vs. observed

The "expected" column records what someone might reasonably assume from the column names. The test shows whether the data confirms it.

In [17]:
EXPECTED = {
    "R1": "one-to-many",   # one customer, many orders (typical retail assumption)
    "R2": "one-to-many", "R3": "one-to-many",
    "R4": "one-to-one",    # one review per order (common assumption)
    "R5": "many-to-one", "R6": "many-to-one", "R7": "many-to-one",
}
card_check = card_df.loc[list(EXPECTED), ["relationship", "cardinality"]].assign(
    expected=pd.Series(EXPECTED),
    confirmed=lambda d: d["cardinality"] == d["expected"])
display(card_check)

,relationship,cardinality,expected,confirmed
id,,,,
R1,customers ↔ orders,one-to-one,one-to-many,False
R2,orders ↔ order_items,one-to-many,one-to-many,True
R3,orders ↔ payments,one-to-many,one-to-many,True
R4,orders ↔ reviews,one-to-many,one-to-one,False
R5,order_items ↔ products,many-to-one,many-to-one,True
R6,order_items ↔ sellers,many-to-one,many-to-one,True
R7,products ↔ translation,many-to-one,many-to-one,True


Two assumptions fail, and both matter:

- **R1 is one-to-one, not one-to-many.** Every `customer_id` places exactly one order, so `customer_id` behaves as an *order-level* customer key. The repeat-customer relationship lives in `customer_unique_id` (Section 5.2).
- **R4 is one-to-many, not one-to-one.** Some orders have several reviews (Section 5.1).

## 5. Special cases

Each case is investigated and documented. **Nothing is removed or corrected.**

### 5.1 Non-unique `review_id`

In [18]:
reviews = datasets["reviews"]
rid_counts = reviews["review_id"].value_counts()
repeated_ids = rid_counts[rid_counts > 1]
rev_dup = reviews[reviews["review_id"].isin(repeated_ids.index)]

print(f"review_id values appearing more than once : {len(repeated_ids):,}")
print(f"rows carrying a repeated review_id        : {len(rev_dup):,}")
print(f"max rows per review_id                    : {rid_counts.max()}")
print(f"(review_id, order_id) duplicated          : {reviews.duplicated(['review_id', 'order_id']).sum()}")

review_id values appearing more than once : 789
rows carrying a repeated review_id        : 1,603
max rows per review_id                    : 3
(review_id, order_id) duplicated          : 0


In [19]:
# Within a repeated review_id, are the review attributes identical across its rows?
content_cols = ["review_score", "review_comment_title", "review_comment_message",
                "review_creation_date", "review_answer_timestamp"]
consistency = (rev_dup.groupby("review_id")[content_cols].nunique(dropna=False) == 1).mean().mul(100).round(2)
display(consistency.rename("% of repeated review_ids with identical values").to_frame())

# Do the orders that share a review_id belong to the same person (customer_unique_id)?
cust = datasets["customers"][["customer_id", "customer_unique_id"]]
shared = (rev_dup[["review_id", "order_id"]]
          .merge(orders[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id", how="left", validate="many_to_one")
          .merge(cust, on="customer_id", how="left", validate="many_to_one"))
per_review = shared.groupby("review_id").agg(n_orders=("order_id", "nunique"),
                                              n_persons=("customer_unique_id", "nunique"),
                                              n_purchase_ts=("order_purchase_timestamp", "nunique"))
print(f"repeated review_ids whose orders all belong to one customer_unique_id: "
      f"{(per_review['n_persons'] == 1).mean():.2%}")
print(f"repeated review_ids whose orders share one purchase timestamp       : "
      f"{(per_review['n_purchase_ts'] == 1).mean():.2%}")

# Time spread between the purchases sharing a review (temporary parsed Series; source column untouched)
purchase_ts = pd.to_datetime(shared["order_purchase_timestamp"])
spread_s = purchase_ts.groupby(shared["review_id"]).agg(lambda s: (s.max() - s.min()).total_seconds())
print(f"... whose purchases fall within 60 seconds of each other           : {(spread_s <= 60).mean():.2%}")
print(f"max purchase-time spread within one review_id                       : {pd.Timedelta(seconds=spread_s.max())}")
display(shared.sort_values(["review_id", "order_id"]).head(6))

,% of repeated review_ids with identical values
review_score,100.0
review_comment_title,100.0
review_comment_message,100.0
review_creation_date,100.0
review_answer_timestamp,100.0


repeated review_ids whose orders all belong to one customer_unique_id: 100.00%


repeated review_ids whose orders share one purchase timestamp       : 32.19%
... whose purchases fall within 60 seconds of each other           : 92.52%
max purchase-time spread within one review_id                       : 127 days 03:24:15


,review_id,order_id,customer_id,order_purchase_timestamp,customer_unique_id
464,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,fef2e5e63da9f3e1dd89e8e319468657,2018-02-02 18:01:10,f30856ad31d3e74253a3f4ccef670648
752,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,a7026133ddbd2e86c83ecd4dfa4dbe01,2018-02-02 18:01:08,f30856ad31d3e74253a3f4ccef670648
1017,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,3bb15be93abf3c2a684d163ba6faaa5b,2017-09-04 15:17:11,91ffaf5eb7f4bd6a48b07f3546fb7a99
1460,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,d39c97549a412a82dacf30fe5f144cb0,2017-09-04 15:17:12,91ffaf5eb7f4bd6a48b07f3546fb7a99
930,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,5043e611d34f6ad724ca658aa22cc43c,2018-02-15 13:09:28,ba16ab3721ffd6b57fc87a7a4f6a4fcf
1498,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,ef9e24d18ac44872aa39e9bb721f9880,2018-02-15 13:09:29,ba16ab3721ffd6b57fc87a7a4f6a4fcf


In [20]:
# The reverse: orders with more than one review
oid_counts = reviews["order_id"].value_counts()
multi_review_orders = oid_counts[oid_counts > 1]
print(f"orders with >1 review: {len(multi_review_orders):,} (max {oid_counts.max()})")
ex = reviews[reviews["order_id"] == multi_review_orders.index[0]].sort_values("review_creation_date")
display(ex[["review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp"]])

orders with >1 review: 547 (max 3)


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,2017-07-21 00:00:00,2017-07-26 13:45:15
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,2017-07-22 00:00:00,2017-07-26 13:40:22


**Interpretation (to be confirmed, not acted on).**

- Every repeated `review_id` carries identical content on all its rows. Its orders always belong to **one person**. In 92.5% of cases those orders were placed within 60 seconds of each other, though the widest gap is 127 days.
- For the large majority, this fits a single review shared across orders that one customer placed together, rather than unrelated reviews colliding on an ID. The long-gap cases do not fit this explanation and stay unexplained.
- Separately, some orders received **several distinct reviews**.

Together these make reviews **many-to-many with orders**. `review_id` alone cannot identify a row, but `(review_id, order_id)` can.

### 5.2 Multiple `customer_id` values per `customer_unique_id`

In [21]:
customers = datasets["customers"]
ids_per_person = customers.groupby("customer_unique_id")["customer_id"].nunique()
print(f"distinct customer_id        : {customers['customer_id'].nunique():,}")
print(f"distinct customer_unique_id : {customers['customer_unique_id'].nunique():,}")
print(f"persons with >1 customer_id : {(ids_per_person > 1).sum():,}  (max {ids_per_person.max()})")
display(ids_per_person.value_counts().sort_index().rename_axis("customer_ids_per_person").rename("persons").to_frame().T)

distinct customer_id        : 99,441
distinct customer_unique_id : 96,096
persons with >1 customer_id : 2,997  (max 17)


customer_ids_per_person,1,2,3,4,5,6,7,9,17
persons,93099,2745,203,30,8,6,3,1,1


In [22]:
# Since customer_id ↔ order is one-to-one (R1), customer_ids per person = orders per person.
orders_per_person = (orders[["order_id", "customer_id"]]
                     .merge(cust, on="customer_id", how="left", validate="one_to_one")
                     .groupby("customer_unique_id")["order_id"].nunique())
print("customer_ids per person == orders per person for every person:",
      orders_per_person.sort_index().equals(ids_per_person.sort_index()))

# Do a person's customer_ids carry different locations? (observation only)
loc_var = customers.groupby("customer_unique_id")[["customer_zip_code_prefix", "customer_city", "customer_state"]].nunique()
multi = loc_var[ids_per_person > 1]
display((multi > 1).sum().rename("persons (with >1 customer_id) having >1 distinct value").to_frame())

customer_ids per person == orders per person for every person: True


,persons (with >1 customer_id) having >1 distinct value
customer_zip_code_prefix,250
customer_city,122
customer_state,39


**Interpretation.**

- A new `customer_id` is issued **for every order**. The number of `customer_id`s per person equals the number of orders per person exactly.
- `customer_unique_id` is the only identifier that links one person's repeat purchases.
- Some people's records show different zip codes, cities or states. Location is therefore stored per order, not per person.

### 5.3 Duplicate geolocation records

In [23]:
geo = datasets["geolocation"]
zip_col = "geolocation_zip_code_prefix"
rows_per_zip = geo[zip_col].value_counts()
print(f"rows                                : {len(geo):,}")
print(f"fully duplicated rows               : {geo.duplicated().sum():,}  ({geo.duplicated().mean():.2%})")
print(f"distinct (zip, lat, lng) points     : {len(geo) - geo.duplicated([zip_col, 'geolocation_lat', 'geolocation_lng']).sum():,}")
print(f"distinct zip prefixes               : {geo[zip_col].nunique():,}")
print(f"zip prefixes with >1 state          : {(geo.groupby(zip_col)['geolocation_state'].nunique() > 1).sum():,}")
print(f"zip prefixes with >1 city spelling  : {(geo.groupby(zip_col)['geolocation_city'].nunique() > 1).sum():,}")
display(rows_per_zip.describe(percentiles=[.5, .9, .99]).round(1).rename("rows per zip prefix").to_frame().T)

rows                                : 1,000,163


fully duplicated rows               : 261,831  (26.18%)


distinct (zip, lat, lng) points     : 720,154
distinct zip prefixes               : 19,015
zip prefixes with >1 state          : 8
zip prefixes with >1 city spelling  : 8,556


,count,mean,std,min,50%,90%,99%,max
rows per zip prefix,19015.0,52.6,72.1,1.0,29.0,124.0,353.0,1146.0


In [24]:
# Examples: one zip prefix whose rows span several states, and a zip prefix's city spellings
multi_state_zip = geo.groupby(zip_col)["geolocation_state"].nunique().loc[lambda s: s > 1].index[0]
display(geo[geo[zip_col] == multi_state_zip].drop_duplicates()[[zip_col, "geolocation_city", "geolocation_state"]]
        .value_counts().rename("rows").to_frame())

rows
geolocation_zip_code_prefix geolocation_city geolocation_state      
2116                        sao paulo        SP                    7
                            são paulo        SP                    3
                                             RN                    1

**Interpretation.**

- `geolocation` is a **point-level** table, not a zip-level lookup. Each zip prefix has many coordinate points (median 29, max 1,146), and about a quarter of all rows are exact repeats.
- City names come in several spellings (for example with and without accents).
- A few zip prefixes appear under more than one state.

None of this is changed here. Choosing how to reduce the table to one row per zip is a cleaning-phase decision.

### 5.4 Composite keys in `order_items` and `payments`

The composite keys from Section 2 are valid. Below, the **sequence numbers** inside them are checked to see whether they behave like a gap-free 1…n count per order. If they do, the highest number equals the number of rows.

In [25]:
def sequence_profile(table: str, seq_col: str) -> dict:
    g = datasets[table].groupby("order_id")[seq_col].agg(["min", "max", "count"])
    return {"table": table, "sequence_column": seq_col, "orders": len(g),
            "orders_with_multiple_rows": int((g["count"] > 1).sum()),
            "max_rows_per_order": int(g["count"].max()),
            "orders_not_starting_at_1": int((g["min"] != 1).sum()),
            "orders_with_gaps (max != count)": int((g["max"] != g["count"]).sum())}

seq_df = pd.DataFrame([sequence_profile("order_items", "order_item_id"),
                       sequence_profile("payments", "payment_sequential")]).set_index("table")
display(seq_df)

,sequence_column,orders,orders_with_multiple_rows,max_rows_per_order,orders_not_starting_at_1,orders_with_gaps (max != count)
table,,,,,,
order_items,order_item_id,98666,9803,21,0,0
payments,payment_sequential,99440,2961,29,80,80


In [26]:
# Example payment sequences that do not start at 1 (observation only)
payments = datasets["payments"]
pg = payments.groupby("order_id")["payment_sequential"].agg(["min", "max", "count"])
gap_orders = pg[pg["min"] != 1].index
display(payments[payments["order_id"].isin(gap_orders[:3])].sort_values(["order_id", "payment_sequential"]))
display(payments[payments["order_id"].isin(gap_orders)]["payment_type"].value_counts().rename("payment rows in gap orders").to_frame())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
27191,00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
12785,056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
40318,0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74


,payment rows in gap orders
payment_type,
debit_card,52
credit_card,29
boleto,1


In [27]:
# order_items: one row per unit? — the same product repeated across order_item_id rows within one order
items = datasets["order_items"]
same_prod = items.groupby(["order_id", "product_id"]).size()
print(f"(order, product) pairs appearing on >1 item row : {(same_prod > 1).sum():,}  (max {same_prod.max()} rows)")
print(f"orders containing >1 distinct seller            : {(items.groupby('order_id')['seller_id'].nunique() > 1).sum():,}")
print(f"orders with >1 item row AND >1 payment row      : "
      f"{len(set(items.loc[items['order_id'].duplicated(), 'order_id']) & set(payments.loc[payments['order_id'].duplicated(), 'order_id'])):,}")

(order, product) pairs appearing on >1 item row : 7,088  (max 20 rows)
orders containing >1 distinct seller            : 1,278
orders with >1 item row AND >1 payment row      : 275


**Interpretation.**

- `order_item_id` is a clean 1…n line counter in every order.
- Quantity is not stored as a column. Buying the same product several times shows up as **repeated rows** (up to 20 for one product in one order). Unit counts must be derived by counting rows.
- `payment_sequential` is **not** always a clean counter. In 80 orders it does not start at 1, mostly where the payment type is debit card. This does not weaken the composite key (it is still unique), but it means `max(payment_sequential)` cannot be used as "number of payments".

### 5.5 Missing product-category translations

In [28]:
cat = products["product_category_name"]
trans_keys = translation["product_category_name"]
print(f"products                                : {len(products):,}")
print(f"  with missing category                 : {cat.isna().sum():,}")
print(f"  with category but no translation      : {untranslated_mask.sum():,}")
print(f"  translatable                          : {(cat.notna() & cat.isin(trans_keys)).sum():,}")
print(f"distinct categories in products         : {cat.nunique()}")
print(f"categories in translation table         : {trans_keys.nunique()}")
print(f"translation rows never used by products : {(~trans_keys.isin(cat)).sum()}")

# Impact on sales lines (how many order_items carry an untranslated or missing category)
item_cat = items[["product_id"]].merge(products[["product_id", "product_category_name"]], on="product_id",
                                       how="left", validate="many_to_one")["product_category_name"]
print(f"\norder_items with missing category       : {item_cat.isna().sum():,}")
print(f"order_items with untranslated category  : {(item_cat.notna() & ~item_cat.isin(trans_keys)).sum():,}")

products                                : 32,951
  with missing category                 : 610
  with category but no translation      : 13
  translatable                          : 32,328
distinct categories in products         : 73
categories in translation table         : 71


translation rows never used by products : 0

order_items with missing category       : 1,603
order_items with untranslated category  : 24


## 6. Join validation

Controlled join experiments measure what really happens when tables are combined.

**Method**

- Every join is a **temporary, in-memory** DataFrame. It is measured and then discarded. Nothing is saved.
- pandas' `merge(validate=...)` is used as a cardinality check. It raises `MergeError` when the data breaks the declared relationship. Each join is tested against the **naive assumption** (what an analyst might take for granted) and against the **observed** cardinality from Section 4.
- `indicator=True` counts how many rows found a partner.
- **Fan-out** is the ratio of output rows to left rows. A fan-out above 1 means left rows have been duplicated.

In [29]:
def try_validate(left, right, on, how, validate):
    try:
        left.merge(right, on=on, how=how, validate=validate)
        return "PASS"
    except pd.errors.MergeError:
        return "FAIL"

JOIN_EXPERIMENTS = [  # (id, left, right, key, naive assumption, observed)
    ("J1", "orders",      "customers",   "customer_id",           "one_to_one",  "one_to_one"),
    ("J2", "orders",      "order_items", "order_id",              "one_to_many", "one_to_many"),
    ("J3", "orders",      "payments",    "order_id",              "one_to_one",  "one_to_many"),
    ("J4", "orders",      "reviews",     "order_id",              "one_to_one",  "one_to_many"),
    ("J5", "order_items", "products",    "product_id",            "many_to_one", "many_to_one"),
    ("J6", "order_items", "sellers",     "seller_id",             "many_to_one", "many_to_one"),
    ("J7", "products",    "translation", "product_category_name", "many_to_one", "many_to_one"),
    ("J8", "order_items", "payments",    "order_id",              "many_to_one", "many_to_many"),
]

join_rows = []
for jid, L, R, key, naive, observed in JOIN_EXPERIMENTS:
    left, right = datasets[L], datasets[R]
    out = left.merge(right, on=key, how="left", indicator=True)
    left_pk = PRIMARY_KEYS[L]
    join_rows.append({
        "id": jid, "join": f"{L} ⟕ {R} on {key}",
        "left_rows": len(left), "output_rows": len(out),
        "rows_added": len(out) - len(left),
        "fan_out": round(len(out) / len(left), 4),
        "left_keys_duplicated": int(out.duplicated(subset=left_pk).sum()),
        "left_rows_unmatched": int((out["_merge"] == "left_only").sum()),
        "naive_assumption": naive, "naive_valid": try_validate(left, right, key, "left", naive),
        "observed": observed, "observed_valid": try_validate(left, right, key, "left", observed),
    })
    del out

join_df = pd.DataFrame(join_rows).set_index("id")
display(join_df)

,join,left_rows,output_rows,rows_added,fan_out,left_keys_duplicated,left_rows_unmatched,naive_assumption,naive_valid,observed,observed_valid
id,,,,,,,,,,,
J1,orders ⟕ customers on customer_id,99441,99441,0,1.0000,0,0,one_to_one,PASS,one_to_one,PASS
J2,orders ⟕ order_items on order_id,99441,113425,13984,1.1406,13984,775,one_to_many,PASS,one_to_many,PASS
J3,orders ⟕ payments on order_id,99441,103887,4446,1.0447,4446,1,one_to_one,FAIL,one_to_many,PASS
J4,orders ⟕ reviews on order_id,99441,99992,551,1.0055,551,768,one_to_one,FAIL,one_to_many,PASS
J5,order_items ⟕ products on product_id,112650,112650,0,1.0000,0,0,many_to_one,PASS,many_to_one,PASS
J6,order_items ⟕ sellers on seller_id,112650,112650,0,1.0000,0,0,many_to_one,PASS,many_to_one,PASS
J7,products ⟕ translation on product_category_name,32951,32951,0,1.0000,0,623,many_to_one,PASS,many_to_one,PASS
J8,order_items ⟕ payments on order_id,112650,117604,4954,1.0440,4954,3,many_to_one,FAIL,many_to_many,PASS


### 6.1 Chained joins: compounding fan-out

Analytical datasets are often built by chaining joins from `orders`. When two one-to-many tables (`order_items` and `payments`) are attached to the same order, their rows **multiply** within each order.

In [30]:
chain = [("orders", None)]
step = datasets["orders"]
chain_rows = [{"step": "orders", "rows": len(step), "distinct order_id": step["order_id"].nunique()}]
for R in ["order_items", "payments", "reviews"]:
    step = step.merge(datasets[R], on="order_id", how="left")
    chain_rows.append({"step": f"+ {R}", "rows": len(step), "distinct order_id": step["order_id"].nunique()})
chain_df = pd.DataFrame(chain_rows)
chain_df["rows_vs_orders"] = (chain_df["rows"] / len(datasets["orders"])).round(3)
display(chain_df)
del step

,step,rows,distinct order_id,rows_vs_orders
0,orders,99441,99441,1.000
1,+ order_items,113425,99441,1.141
2,+ payments,118434,99441,1.191
3,+ reviews,119143,99441,1.198


### 6.2 Impact on measures: double counting

The key risk is not the row count itself but **inflated sums**. This compares the true totals with the totals computed after a naive `order_items ⋈ payments` join.

In [31]:
naive = items.merge(payments, on="order_id", how="inner")
impact = pd.DataFrame({
    "true_total": {"order_items.price": items["price"].sum(),
                   "order_items.freight_value": items["freight_value"].sum(),
                   "payments.payment_value": payments["payment_value"].sum()},
    "total_after_naive_join": {"order_items.price": naive["price"].sum(),
                               "order_items.freight_value": naive["freight_value"].sum(),
                               "payments.payment_value": naive["payment_value"].sum()},
})
impact["inflation_pct"] = ((impact["total_after_naive_join"] / impact["true_total"] - 1) * 100).round(2)
display(impact.round(2))
print(f"rows: order_items={len(items):,}  payments={len(payments):,}  naive join={len(naive):,}")
del naive

,true_total,total_after_naive_join,inflation_pct
order_items.price,13591643.70,14209115.34,4.54
order_items.freight_value,2251909.54,2357428.51,4.69
payments.payment_value,16008872.12,20308134.71,26.86


rows: order_items=112,650  payments=103,886  naive join=117,601


### 6.3 Geolocation joins (estimated without materialising)

Because a zip prefix has many geolocation rows, joining customers or sellers to `geolocation` on zip multiplies every row by the number of points for that zip. The result size is **calculated** from key counts rather than built: for each zip, left rows × geolocation rows, plus one row for each unmatched left row. Building it would need many millions of rows for no purpose.

In [32]:
geo_counts = geo[zip_col].value_counts()
geo_join = []
for L, k in [("customers", "customer_zip_code_prefix"), ("sellers", "seller_zip_code_prefix")]:
    per_row = datasets[L][k].map(geo_counts).fillna(1)  # temporary Series for counting; table untouched
    geo_join.append({"join": f"{L} ⟕ geolocation on zip", "left_rows": len(datasets[L]),
                     "estimated_output_rows": int(per_row.sum()),
                     "fan_out": round(per_row.sum() / len(datasets[L]), 1),
                     "left_rows_unmatched": int((~datasets[L][k].isin(geo_counts.index)).sum())})
display(pd.DataFrame(geo_join).set_index("join"))

,left_rows,estimated_output_rows,fan_out,left_rows_unmatched
join,,,,
customers ⟕ geolocation on zip,99441,15083733,151.7,278
sellers ⟕ geolocation on zip,3095,435094,140.6,7


### 6.4 Implications for future analytical datasets

| Finding | Implication |
|---|---|
| `orders ⟕ order_items` fans out (by design) | The result is at **order-line grain**. Order-level measures (e.g. delivery dates) repeat on every line, so they must not be summed or averaged across lines without deduplication. |
| `payments` and `reviews` are one-to-many per order | Aggregate them to **one row per order** (e.g. total paid, number of payments, a chosen review) *before* joining to orders or items. |
| `order_items ⋈ payments` is many-to-many | Never join these directly. Both price and payment totals become inflated (6.2). Bring them together only at a common grain, i.e. the order. |
| Chained joins multiply | Every additional one-to-many table multiplies the rows again (6.1). Decide the target grain first, then aggregate each table to that grain. |
| Joins to products, sellers, customers and translation are safe | They are validated many-to-one / one-to-one and keep the row count unchanged. |
| `geolocation` has no key | It cannot be joined directly. It must first be reduced to one row per zip prefix (a cleaning-phase decision); otherwise the fan-out is massive (6.3). |
| Inner vs. left joins | Inner joins silently drop orders with no items (775), no payment (1) or no review (768), and products without a translation. The choice of join type changes the population being analysed. |
| `customer_id` is per order | Customer-level analytics (retention, RFM, CLV) must use `customer_unique_id`. |
| Use `validate=` in pipelines | Future integration code should declare the expected cardinality on every merge so that any regression fails loudly. |

## 7. Entity-relationship diagram

The ERD is **generated from the test results above**. Each crow's-foot symbol is computed from the measured minimum and maximum number of matching rows on each side, and the PK markers come from the validated keys in Section 2.3. Hand-drawn symbols could drift from the data; computed ones cannot.

Reading the notation (Mermaid `erDiagram`): `||` exactly one · `o|` zero or one · `|{` one or many · `o{` zero or many. Each symbol describes how many rows of the table **next to it** relate to one row of the table at the other end.

In [33]:
def participation(L, lk, R, rk):
    # (min, max) rows of L per row of R, and (min, max) rows of R per row of L.
    l, r = datasets[L][lk], datasets[R][rk]
    lc, rc = l.value_counts(), r.value_counts()
    L_per_R = r.map(lc).fillna(0)          # for each R row: how many L rows match
    R_per_L = l.map(rc).fillna(0)          # for each L row: how many R rows match
    return (int(L_per_R.min()), int(L_per_R.max())), (int(R_per_L.min()), int(R_per_L.max()))

def sym(mn, mx, left_side: bool):
    if left_side:
        return {(0, False): "|o", (1, False): "||", (0, True): "}o", (1, True): "}|"}[(min(mn, 1), mx > 1)]
    return {(0, False): "o|", (1, False): "||", (0, True): "o{", (1, True): "|{"}[(min(mn, 1), mx > 1)]

ERD_EDGES = [  # (L, lk, R, rk, label) — L is drawn on the left
    ("customers",   "customer_id",                 "orders",      "customer_id",              "places"),
    ("orders",      "order_id",                    "order_items", "order_id",                 "contains"),
    ("orders",      "order_id",                    "payments",    "order_id",                 "paid_by"),
    ("orders",      "order_id",                    "reviews",     "order_id",                 "reviewed_in"),
    ("products",    "product_id",                  "order_items", "product_id",               "sold_as"),
    ("sellers",     "seller_id",                   "order_items", "seller_id",                "fulfils"),
    ("translation", "product_category_name",       "products",    "product_category_name",    "translates"),
    ("geolocation", "geolocation_zip_code_prefix", "customers",   "customer_zip_code_prefix", "zip prefix (no key)"),
    ("geolocation", "geolocation_zip_code_prefix", "sellers",     "seller_zip_code_prefix",   "zip prefix (no key)"),
]

ERD_COLUMNS = {
    "customers":   ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
    "orders":      ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_delivered_customer_date"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
    "payments":    ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"],
    "reviews":     ["review_id", "order_id", "review_score", "review_creation_date"],
    "products":    ["product_id", "product_category_name", "product_weight_g"],
    "sellers":     ["seller_id", "seller_zip_code_prefix", "seller_state"],
    "translation": ["product_category_name", "product_category_name_english"],
    "geolocation": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_state"],
}
FK_COLS = {(c, ck) for _, _, _, _, c, ck in RELATIONSHIPS[:7]}
NOTES = {("reviews", "review_id"): "not unique alone", ("customers", "customer_unique_id"): "person-level id",
         ("payments", "payment_sequential"): "80 orders do not start at 1",
         ("geolocation", "geolocation_zip_code_prefix"): "not unique - no PK"}

def mtype(dtype) -> str:
    s = str(dtype)
    return "int" if s.startswith("int") else "float" if s.startswith("float") else "string"

lines = ["erDiagram"]
for L, lk, R, rk, label in ERD_EDGES:
    (lmin, lmax), (rmin, rmax) = participation(L, lk, R, rk)
    lines.append(f'    {L.upper()} {sym(lmin, lmax, True)}--{sym(rmin, rmax, False)} {R.upper()} : "{label}"')
for t, cols in ERD_COLUMNS.items():
    lines.append(f"    {t.upper()} {{")
    for c in cols:
        keys = [k for k, cond in [("PK", c in (PRIMARY_KEYS[t] or [])), ("FK", (t, c) in FK_COLS)] if cond]
        note = f' "{NOTES[(t, c)]}"' if (t, c) in NOTES else ""
        lines.append(f"        {mtype(datasets[t][c].dtype)} {c}{' ' + ', '.join(keys) if keys else ''}{note}")
    lines.append("    }")
ERD_MERMAID = "\n".join(lines)

display(Markdown(f"```mermaid\n{ERD_MERMAID}\n```"))

```mermaid
erDiagram
    CUSTOMERS ||--|| ORDERS : "places"
    ORDERS ||--o{ ORDER_ITEMS : "contains"
    ORDERS ||--o{ PAYMENTS : "paid_by"
    ORDERS ||--o{ REVIEWS : "reviewed_in"
    PRODUCTS ||--|{ ORDER_ITEMS : "sold_as"
    SELLERS ||--|{ ORDER_ITEMS : "fulfils"
    TRANSLATION |o--|{ PRODUCTS : "translates"
    GEOLOCATION }o--o{ CUSTOMERS : "zip prefix (no key)"
    GEOLOCATION }o--o{ SELLERS : "zip prefix (no key)"
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id "person-level id"
        int customer_zip_code_prefix
        string customer_city
        string customer_state
    }
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        string order_purchase_timestamp
        string order_delivered_customer_date
    }
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        float price
        float freight_value
    }
    PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK "80 orders do not start at 1"
        string payment_type
        int payment_installments
        float payment_value
    }
    REVIEWS {
        string review_id PK "not unique alone"
        string order_id PK, FK
        int review_score
        string review_creation_date
    }
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        float product_weight_g
    }
    SELLERS {
        string seller_id PK
        int seller_zip_code_prefix
        string seller_state
    }
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }
    GEOLOCATION {
        int geolocation_zip_code_prefix "not unique - no PK"
        float geolocation_lat
        float geolocation_lng
        string geolocation_state
    }
```

In [34]:
print(ERD_MERMAID)  # raw source, for copying into docs or https://mermaid.live

erDiagram
    CUSTOMERS ||--|| ORDERS : "places"
    ORDERS ||--o{ ORDER_ITEMS : "contains"
    ORDERS ||--o{ PAYMENTS : "paid_by"
    ORDERS ||--o{ REVIEWS : "reviewed_in"
    PRODUCTS ||--|{ ORDER_ITEMS : "sold_as"
    SELLERS ||--|{ ORDER_ITEMS : "fulfils"
    TRANSLATION |o--|{ PRODUCTS : "translates"
    GEOLOCATION }o--o{ CUSTOMERS : "zip prefix (no key)"
    GEOLOCATION }o--o{ SELLERS : "zip prefix (no key)"
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id "person-level id"
        int customer_zip_code_prefix
        string customer_city
        string customer_state
    }
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        string order_purchase_timestamp
        string order_delivered_customer_date
    }
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        float price
        float freight_value
    }
   

**Static copy of the generated diagram.** This renders in viewers that support Mermaid in markdown cells, such as GitHub, JupyterLab ≥ 4.1 and VS Code. The cell below confirms it matches the generated version.

```mermaid
erDiagram
    CUSTOMERS ||--|| ORDERS : "places"
    ORDERS ||--o{ ORDER_ITEMS : "contains"
    ORDERS ||--o{ PAYMENTS : "paid_by"
    ORDERS ||--o{ REVIEWS : "reviewed_in"
    PRODUCTS ||--|{ ORDER_ITEMS : "sold_as"
    SELLERS ||--|{ ORDER_ITEMS : "fulfils"
    TRANSLATION |o--|{ PRODUCTS : "translates"
    GEOLOCATION }o--o{ CUSTOMERS : "zip prefix (no key)"
    GEOLOCATION }o--o{ SELLERS : "zip prefix (no key)"
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id "person-level id"
        int customer_zip_code_prefix
        string customer_city
        string customer_state
    }
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        string order_purchase_timestamp
        string order_delivered_customer_date
    }
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        float price
        float freight_value
    }
    PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK "80 orders do not start at 1"
        string payment_type
        int payment_installments
        float payment_value
    }
    REVIEWS {
        string review_id PK "not unique alone"
        string order_id PK, FK
        int review_score
        string review_creation_date
    }
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        float product_weight_g
    }
    SELLERS {
        string seller_id PK
        int seller_zip_code_prefix
        string seller_state
    }
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }
    GEOLOCATION {
        int geolocation_zip_code_prefix "not unique - no PK"
        float geolocation_lat
        float geolocation_lng
        string geolocation_state
    }
```

In [35]:
STATIC_ERD = '''erDiagram
    CUSTOMERS ||--|| ORDERS : "places"
    ORDERS ||--o{ ORDER_ITEMS : "contains"
    ORDERS ||--o{ PAYMENTS : "paid_by"
    ORDERS ||--o{ REVIEWS : "reviewed_in"
    PRODUCTS ||--|{ ORDER_ITEMS : "sold_as"
    SELLERS ||--|{ ORDER_ITEMS : "fulfils"
    TRANSLATION |o--|{ PRODUCTS : "translates"
    GEOLOCATION }o--o{ CUSTOMERS : "zip prefix (no key)"
    GEOLOCATION }o--o{ SELLERS : "zip prefix (no key)"
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id "person-level id"
        int customer_zip_code_prefix
        string customer_city
        string customer_state
    }
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        string order_purchase_timestamp
        string order_delivered_customer_date
    }
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        float price
        float freight_value
    }
    PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK "80 orders do not start at 1"
        string payment_type
        int payment_installments
        float payment_value
    }
    REVIEWS {
        string review_id PK "not unique alone"
        string order_id PK, FK
        int review_score
        string review_creation_date
    }
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        float product_weight_g
    }
    SELLERS {
        string seller_id PK
        int seller_zip_code_prefix
        string seller_state
    }
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }
    GEOLOCATION {
        int geolocation_zip_code_prefix "not unique - no PK"
        float geolocation_lat
        float geolocation_lng
        string geolocation_state
    }'''
assert STATIC_ERD.strip() == ERD_MERMAID.strip(), 'Static ERD copy is out of date with the data-derived ERD'
print('Static ERD matches the data-derived ERD.')

Static ERD matches the data-derived ERD.


## 8. Final summary

### 8.1 Primary key assessment

In [36]:
display(pk_assessment_df[["candidate_key", "type", "total_records", "unique_values", "duplicated_values",
                          "missing_values", "uniqueness_pct", "valid_pk"]])

,candidate_key,type,total_records,unique_values,duplicated_values,missing_values,uniqueness_pct,valid_pk
table,,,,,,,,
customers,customer_id,single,99441,99441,0,0,100.0000,True
geolocation,(none — all columns tested),none,1000163,738332,261831,0,73.8212,False
orders,order_id,single,99441,99441,0,0,100.0000,True
order_items,order_id + order_item_id,composite,112650,112650,0,0,100.0000,True
payments,order_id + payment_sequential,composite,103886,103886,0,0,100.0000,True
reviews,review_id + order_id,composite,99224,99224,0,0,100.0000,True
products,product_id,single,32951,32951,0,0,100.0000,True
sellers,seller_id,single,3095,3095,0,0,100.0000,True
translation,product_category_name,single,71,71,0,0,100.0000,True


### 8.2 Foreign key validation

In [37]:
display(fk_df[["relationship", "child → parent", "matched_records", "unmatched_records", "child_missing",
               "integrity_pct", "distinct_child_ids", "distinct_ids_referenced", "parent_coverage_pct"]])

,relationship,child → parent,matched_records,unmatched_records,child_missing,integrity_pct,distinct_child_ids,distinct_ids_referenced,parent_coverage_pct
id,,,,,,,,,
R1,customers ↔ orders,orders.customer_id → customers.customer_id,99441,0,0,100.0000,99441,99441,100.0000
R2,orders ↔ order_items,order_items.order_id → orders.order_id,112650,0,0,100.0000,98666,98666,99.2206
R3,orders ↔ payments,payments.order_id → orders.order_id,103886,0,0,100.0000,99440,99440,99.9990
R4,orders ↔ reviews,reviews.order_id → orders.order_id,99224,0,0,100.0000,98673,98673,99.2277
R5,order_items ↔ products,order_items.product_id → products.product_id,112650,0,0,100.0000,32951,32951,100.0000
R6,order_items ↔ sellers,order_items.seller_id → sellers.seller_id,112650,0,0,100.0000,3095,3095,100.0000
R7,products ↔ translation,products.product_category_name → translation.product_category_name,32328,13,610,99.9598,73,71,100.0000
S1,customers ↔ geolocation (zip),customers.customer_zip_code_prefix → geolocation.geolocation_zip_code_prefix,99163,278,0,99.7204,14994,14837,78.0279
S2,sellers ↔ geolocation (zip),sellers.seller_zip_code_prefix → geolocation.geolocation_zip_code_prefix,3088,7,0,99.7738,2246,2239,11.7749


### 8.3 Relationship cardinality

In [38]:
display(card_df[["relationship", "join_key", "max_L_per_key", "max_R_per_key", "L_keys_without_R",
                 "R_rows_without_L", "cardinality"]])

,relationship,join_key,max_L_per_key,max_R_per_key,L_keys_without_R,R_rows_without_L,cardinality
id,,,,,,,
R1,customers ↔ orders,customer_id,1,1,0,0,one-to-one
R2,orders ↔ order_items,order_id,1,21,775,0,one-to-many
R3,orders ↔ payments,order_id,1,29,1,0,one-to-many
R4,orders ↔ reviews,order_id,1,3,768,0,one-to-many
R5,order_items ↔ products,product_id,527,1,0,0,many-to-one
R6,order_items ↔ sellers,seller_id,2033,1,0,0,many-to-one
R7,products ↔ translation,product_category_name,3029,1,2,0,many-to-one
S1,customers ↔ geolocation,customer_zip_code_prefix = geolocation_zip_code_prefix,142,1146,157,31864,many-to-many
S2,sellers ↔ geolocation,seller_zip_code_prefix = geolocation_zip_code_prefix,49,965,7,731347,many-to-many


### 8.4 ERD

See Section 7. The diagram is generated from the measured cardinalities and validated keys.

### 8.5 Referential integrity findings

| # | Relationship | Integrity | Notes |
|---|---|---|---|
| R1 | orders → customers | **100%** | Every order has a customer and every customer has an order |
| R2 | order_items → orders | **100%** | 775 orders have **no items** (603 `unavailable`, 164 `canceled`, 5 `created`, 2 `invoiced`, 1 `shipped`) |
| R3 | payments → orders | **100%** | **1** order has no payment (status `delivered`) |
| R4 | reviews → orders | **100%** | 768 orders have **no review** |
| R5 | order_items → products | **100%** | Every product has been sold at least once |
| R6 | order_items → sellers | **100%** | Every seller has sold at least once |
| R7 | products → translation | **99.96%** | 13 products in **2** categories have no translation (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`). A further **610** products have no category at all. Together these affect 1,627 order lines (24 + 1,603) |
| S1 | customers → geolocation (zip) | 99.72% | 278 customers (157 zip prefixes) have no coordinates. The target is **not unique**, so this is not a true foreign key |
| S2 | sellers → geolocation (zip) | 99.77% | 7 sellers (7 zip prefixes) have no coordinates. Same caveat as S1 |

**Conclusion:**
- The **transactional core** (orders, items, payments, reviews, products, sellers, customers) has **no orphan records**. Every reference resolves.
- The gaps all sit on the **parent side**: orders without items, payments or reviews.
- The exceptions are the missing category translations and the geolocation lookups.

### 8.6 Key findings

1. **Primary keys:**
   - 8 of 9 tables have a valid key. Four are single-column (`customer_id`, `order_id`, `product_id`, `seller_id`) and `translation` uses `product_category_name`.
   - Three tables need a **composite key**: `order_items (order_id, order_item_id)`, `payments (order_id, payment_sequential)` and `reviews (review_id, order_id)`.
   - **`geolocation` has no key**: 261,831 rows are exact duplicates.
2. **`customer_id` is an order-level identifier, not a person.**
   - customers ↔ orders is **one-to-one**.
   - The real person is `customer_unique_id`: 96,096 people, 2,997 of whom have 2–17 `customer_id`s (one per order).
3. **`review_id` is not unique:**
   - 789 IDs appear on 1,603 rows, each linking a single identical review to several orders of the same person.
   - Separately, 547 orders have 2–3 distinct reviews.
   - Reviews are effectively **many-to-many** with orders.
4. **`payment_sequential` has gaps.** 80 orders' sequences do not start at 1, though the composite key is still unique.
5. **Quantity is stored as rows.** The same product repeats across `order_item_id` rows (7,088 order–product pairs, up to 20 rows).
6. **Joins:**
   - Joins to dimensions (customers, products, sellers, translation) are **safe**: validated one-to-one or many-to-one, with no change in row count.
   - Joins to one-to-many tables increase row counts: payments +4.5%, reviews +0.6%, items +14.1%.

### 8.7 Important warnings for future data integration

> ⚠️ **1. Never join `order_items` and `payments` directly.**
> They are many-to-many through `order_id`. A naive join inflates **payment value by 26.86%** and item **price by 4.54%** (freight by 4.69%). Aggregate payments to one row per order first.

> ⚠️ **2. Decide the grain before joining, and aggregate to it.**
> Chaining orders + items + payments + reviews turns 99,441 orders into 119,143 rows. Order-level fields such as delivery dates or payment totals are then repeated and get double-counted.

> ⚠️ **3. Use `customer_unique_id` for any customer-level analysis** (repeat purchase, retention, RFM, CLV, segmentation). Using `customer_id` would make every customer look like a one-time buyer.

> ⚠️ **4. Do not assume one review per order.** Decide on a rule (e.g. the latest review, or the mean score) before attaching reviews to orders, and keep in mind that one review can belong to several orders.

> ⚠️ **5. `geolocation` cannot be joined as-is.** Joining on zip prefix would turn 99,441 customers into about **15.1 million rows** (×151.7). The table must first be reduced to one row per zip prefix. Also expect 278 customers and 7 sellers with no coordinates, a few zip prefixes spanning several states, and many city spelling variants.

> ⚠️ **6. Zip prefixes were read as integers** (see notebook 01). The joins work because both sides are integers, but leading zeros are lost. Load them as text, consistently on both sides, before using them as codes.

> ⚠️ **7. Inner joins silently drop records.** Inner-joining orders to items, payments or reviews removes 775, 1 and 768 orders respectively, and inner-joining products to translation removes 623 products. Choose left or inner joins deliberately and record row counts at each step.

> ⚠️ **8. Plan for missing and untranslated categories.** 610 products have no category and 13 have no English translation. Decide how they will be labelled before reporting by category.

> ⚠️ **9. Do not derive payment counts from `payment_sequential`.** Count rows instead, because of the 80 orders whose sequences start above 1.

> ⚠️ **10. Declare cardinality on every merge.** Pass `validate=` to `pd.merge` in all integration code (as in Section 6), so that a change in the data's relationships fails loudly instead of silently duplicating rows.

**Scope note:** this notebook only observed and measured. No data was cleaned, de-duplicated, filled, re-typed or saved.

## 9. Verification

In [39]:
fingerprints_after = pd.DataFrame([fingerprint(RAW_DIR / f) for f in FILES.values()]).set_index("file")
integrity = (fingerprints_before == fingerprints_after)
integrity["untouched"] = integrity.all(axis=1)
display(integrity)

processed_after = sorted(p.name for p in PROCESSED_DIR.iterdir()) if PROCESSED_DIR.exists() else []

checks = {
    "All 9 datasets loaded":                          len(datasets) == 9 and not load_errors,
    "All 9 datasets assessed for a primary key":      set(pk_assessment_df.index) == set(FILES),
    "All 9 datasets appear in the key-discovery scan": set(pk_df["table"]) == set(FILES),
    "All 7 required FK tests executed":               {f"R{i}" for i in range(1, 8)} <= set(fk_df.index),
    "All FK tests (incl. supplementary) error-free":  not fk_errors and len(fk_df) == len(RELATIONSHIPS),
    "All cardinality tests executed":                 len(card_df) == len(CARDINALITY_TESTS),
    "All join experiments executed":                  len(join_df) == len(JOIN_EXPERIMENTS),
    "Observed cardinalities validated by merge()":    (join_df["observed_valid"] == "PASS").all(),
    "ERD generated":                                  ERD_MERMAID.startswith("erDiagram"),
    "Raw files unchanged (size, mtime, SHA-256)":     integrity["untouched"].all(),
    "No files written to data/processed/":            processed_after == processed_before,
}
for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
assert all(checks.values()), "Verification failed — see FAIL lines."
print("\nNotebook completed: all checks passed.")

,size_bytes,modified_ns,sha256,untouched
file,,,,
olist_customers_dataset.csv,True,True,True,True
olist_geolocation_dataset.csv,True,True,True,True
olist_orders_dataset.csv,True,True,True,True
olist_order_items_dataset.csv,True,True,True,True
olist_order_payments_dataset.csv,True,True,True,True
olist_order_reviews_dataset.csv,True,True,True,True
olist_products_dataset.csv,True,True,True,True
olist_sellers_dataset.csv,True,True,True,True
product_category_name_translation.csv,True,True,True,True


[PASS] All 9 datasets loaded
[PASS] All 9 datasets assessed for a primary key
[PASS] All 9 datasets appear in the key-discovery scan
[PASS] All 7 required FK tests executed
[PASS] All FK tests (incl. supplementary) error-free
[PASS] All cardinality tests executed
[PASS] All join experiments executed
[PASS] Observed cardinalities validated by merge()
[PASS] ERD generated
[PASS] Raw files unchanged (size, mtime, SHA-256)
[PASS] No files written to data/processed/

Notebook completed: all checks passed.
